# Sprint 1 — Component 4 Lung Mask Decoder Training

Trains **only** the MedSAM ViT-B mask decoder on Montgomery + Shenzhen lung annotations.
The image encoder and prompt encoder stay completely frozen — only the decoder weights change.

## Datasets to attach before running

| Kaggle dataset slug | Mount path | Purpose |
|---|---|---|
| `iahmedhabib/montgomery` | `/kaggle/input/montgomery` | CXR images + left/right lung masks |
| `iahmedhabib/shehzhenn` | `/kaggle/input/shehzhenn` | CXR images + combined lung masks |
| `iahmedhabib/medsam-vit-b` | `/kaggle/input/medsam-vit-b` | MedSAM ViT-B pretrained checkpoint |

## What this trains

- **Model**: MedSAM ViT-B mask decoder (~4 M trainable params; encoder is frozen)
- **Loss**: BCE + Dice on 256×256 low-res masks
- **Schedule**: 3-epoch linear warmup → cosine decay to floor LR over 60 epochs
- **Augmentation**: horizontal flip, ±10° rotation (image+mask jointly), brightness jitter, Gaussian noise
- **Dataset mixing**: interleaved by dataset before DataLoader shuffle so every batch is mixed

## Expected outcome

| Metric | Before | Expected after |
|---|---|---|
| Montgomery val Dice | 0.42 | 0.65–0.72 |
| Shenzhen val Dice | 0.51 | 0.68–0.75 |

Estimated wall-clock: **~90 minutes on a T4 GPU**.

In [ ]:
# ── Cell 1: Clone repo + install dependencies ────────────────────────────────
import subprocess, sys, os
from pathlib import Path

REPO_URL = 'https://github.com/mabdullahi7780/dl-project-codebase.git'
REPO_DIR = Path('/kaggle/working/repo')

if not REPO_DIR.exists():
    print('Cloning repo...')
    subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, str(REPO_DIR)], check=True)
else:
    print('Repo already exists — pulling latest...')
    subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only'], check=True)

os.chdir(str(REPO_DIR))
sys.path.insert(0, str(REPO_DIR))

print('Installing dependencies...')
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'],
    check=True
)

import torch
print(f'PyTorch {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)} ({torch.cuda.get_device_properties(0).total_memory // 1024**3} GB)')
print(f'Repo branch: ', end='')
subprocess.run(['git', 'branch', '--show-current'], cwd=str(REPO_DIR))

In [ ]:
# ── Cell 2: Discover dataset paths ───────────────────────────────────────────
from pathlib import Path

INPUT = Path('/kaggle/input')

def _find(*candidates):
    """Return the first existing path from the candidate list, or None."""
    for p in candidates:
        if Path(p).exists():
            return Path(p)
    return None

# MedSAM pretrained backbone — required
MEDSAM_CKPT = _find(
    INPUT / 'medsam-vit-b' / 'medsam_vit_b.pth',
    INPUT / 'medsam-vit-b' / 'MedSAM' / 'medsam_vit_b.pth',
    REPO_DIR / 'checkpoints' / 'medsam' / 'medsam_vit_b.pth',
)

# Montgomery dataset root — contains images + left/right lung masks
MONTGOMERY_ROOT = _find(
    INPUT / 'montgomery' / 'MontgomerySet',
    INPUT / 'montgomery' / 'montgomery' / 'MontgomerySet',
    INPUT / 'montgomery',
)

# Shenzhen dataset root — contains images + combined lung masks
SHENZHEN_ROOT = _find(
    INPUT / 'shehzhenn' / 'ChinaSet_AllFiles',
    INPUT / 'shehzhenn' / 'shenzhen' / 'ChinaSet_AllFiles',
    INPUT / 'shehzhenn',
)

print('Path discovery results:')
print(f'  MEDSAM_CKPT    : {MEDSAM_CKPT or "NOT FOUND"}')
print(f'  MONTGOMERY_ROOT: {MONTGOMERY_ROOT or "NOT FOUND"}')
print(f'  SHENZHEN_ROOT  : {SHENZHEN_ROOT or "NOT FOUND"}')

assert MEDSAM_CKPT is not None, (
    'MedSAM checkpoint not found. Attach the iahmedhabib/medsam-vit-b dataset '
    'and confirm the file is named medsam_vit_b.pth'
)
assert MONTGOMERY_ROOT is not None or SHENZHEN_ROOT is not None, (
    'No lung mask dataset found. Attach at least one of: '
    'iahmedhabib/montgomery, iahmedhabib/shehzhenn'
)

In [ ]:
# ── Cell 3: Build component4_manifest.csv ────────────────────────────────────
#
# The training script expects a CSV with columns:
#   sample_id, image_path, mask_path, dataset, split
#
# Montgomery masks come as separate left/right PNG files — the dataset loader
# merges them on-the-fly when mask_path contains a pipe character (left|right).
# Shenzhen has a single combined mask per image.
#
# Split assignment: ~80% train, ~20% val using a deterministic hash of the
# filename stem so the split is stable across runs.

import csv
from pathlib import Path

OUTPUT_DIR = Path('/kaggle/working/component4_out')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
MANIFEST_PATH = OUTPUT_DIR / 'component4_manifest.csv'

rows = []

def _split(stem: str) -> str:
    """Deterministic 80/20 train/val split by filename hash."""
    return 'val' if abs(hash(stem)) % 5 == 0 else 'train'

# ── Montgomery ──────────────────────────────────────────────────────────────
if MONTGOMERY_ROOT is not None:
    mont_root = Path(MONTGOMERY_ROOT)

    # Images can live at different depths depending on the Kaggle dataset version
    img_dir = next(
        (d for d in [
            mont_root / 'CXR_png',
            mont_root.parent / 'CXR_png',
        ] if d.exists()),
        None
    )

    left_dir = next(
        (d for d in [
            mont_root / 'ManualMask' / 'leftMask',
            mont_root.parent / 'ManualMask' / 'leftMask',
        ] if d.exists()),
        None
    )
    right_dir = next(
        (d for d in [
            mont_root / 'ManualMask' / 'rightMask',
            mont_root.parent / 'ManualMask' / 'rightMask',
        ] if d.exists()),
        None
    )

    if img_dir and left_dir and right_dir:
        for img in sorted(img_dir.glob('*.png')):
            stem = img.stem
            left  = left_dir  / f'{stem}.png'
            right = right_dir / f'{stem}.png'
            if left.exists() and right.exists():
                rows.append({
                    'sample_id' : stem,
                    'image_path': str(img),
                    'mask_path' : f'{left}|{right}',   # merged on-the-fly by the dataset loader
                    'dataset'   : 'montgomery',
                    'split'     : _split(stem),
                })
        n_mont = sum(1 for r in rows if r['dataset'] == 'montgomery')
        print(f'Montgomery: {n_mont} images with masks found')
        print(f'  img_dir   : {img_dir}')
        print(f'  left_dir  : {left_dir}')
        print(f'  right_dir : {right_dir}')
    else:
        missing = [n for n, d in [('img_dir', img_dir), ('left_dir', left_dir), ('right_dir', right_dir)] if d is None]
        print(f'Montgomery: could not locate subdirs: {missing} — skipping')
        print(f'  mont_root contents: {[p.name for p in mont_root.iterdir()]}')

# ── Shenzhen ─────────────────────────────────────────────────────────────────
if SHENZHEN_ROOT is not None:
    shen_root = Path(SHENZHEN_ROOT)

    img_dir = next(
        (d for d in [
            shen_root / 'CXR_png',
            shen_root.parent / 'CXR_png',
        ] if d.exists()),
        None
    )
    mask_dir = next(
        (d for d in [
            shen_root / 'Masks',
            shen_root.parent / 'Masks',
        ] if d.exists()),
        None
    )

    if img_dir and mask_dir:
        n_before = len(rows)
        for img in sorted(img_dir.glob('*.png')):
            stem = img.stem
            # Shenzhen masks are named either {stem}_mask.png or {stem}.png
            mask = mask_dir / f'{stem}_mask.png'
            if not mask.exists():
                mask = mask_dir / f'{stem}.png'
            if mask.exists():
                rows.append({
                    'sample_id' : stem,
                    'image_path': str(img),
                    'mask_path' : str(mask),
                    'dataset'   : 'shenzhen',
                    'split'     : _split(stem),
                })
        n_shen = len(rows) - n_before
        print(f'Shenzhen: {n_shen} images with masks found')
        print(f'  img_dir  : {img_dir}')
        print(f'  mask_dir : {mask_dir}')
    else:
        missing = [n for n, d in [('img_dir', img_dir), ('mask_dir', mask_dir)] if d is None]
        print(f'Shenzhen: could not locate subdirs: {missing} — skipping')
        print(f'  shen_root contents: {[p.name for p in shen_root.iterdir()]}')

# ── Write manifest ────────────────────────────────────────────────────────────
assert rows, 'No image+mask pairs found. Check dataset paths above before continuing.'

with open(MANIFEST_PATH, 'w', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=['sample_id', 'image_path', 'mask_path', 'dataset', 'split'])
    writer.writeheader()
    writer.writerows(rows)

train_rows = [r for r in rows if r['split'] == 'train']
val_rows   = [r for r in rows if r['split'] == 'val']
from collections import Counter
train_counts = Counter(r['dataset'] for r in train_rows)
val_counts   = Counter(r['dataset'] for r in val_rows)

print(f'\nManifest written: {MANIFEST_PATH}')
print(f'  Train: {len(train_rows)} samples — {dict(train_counts)}')
print(f'  Val:   {len(val_rows)} samples — {dict(val_counts)}')

In [ ]:
# ── Cell 4: Train Component 4 ─────────────────────────────────────────────────
#
# What is being trained:
#   Only the MedSAM mask decoder. The ViT-B image encoder (frozen) produces a
#   [B, 256, 64, 64] embedding from each 1024×1024 CXR. The decoder takes that
#   embedding plus a bounding-box prompt and outputs a [B, 1, 256, 256] logit map.
#   We supervise it with BCE + Dice loss against the manual lung masks.
#
# Sprint 1 improvements over the old run:
#   1. Cosine LR schedule: warmup 0→3e-4 for 3 epochs, then cosine decay to 3e-6
#      by epoch 60 — lets the decoder take large steps early and fine-grained
#      boundary steps late, where Dice is actually won or lost.
#   2. More epochs (40→60) and patience (8→15).
#   3. ±10° rotation (applied identically to image AND mask).
#   4. Gaussian noise on the image only (std=0.02).
#   5. Round-robin dataset interleaving: records are pre-sorted so early batches
#      are always mixed (Montgomery+Shenzhen), not all-Montgomery then all-Shenzhen.
#   6. Threshold sweep at the end: finds the optimal binarisation threshold on
#      the val set instead of assuming 0.5.

import subprocess, sys, os
from pathlib import Path

REPO_DIR   = Path('/kaggle/working/repo')
OUTPUT_DIR = Path('/kaggle/working/component4_out')
MANIFEST_PATH = OUTPUT_DIR / 'component4_manifest.csv'

env = os.environ.copy()
env['COMPONENT4_TRAIN_MANIFEST'] = str(MANIFEST_PATH)
env['COMPONENT4_VAL_MANIFEST']   = str(MANIFEST_PATH)
env['COMPONENT4_SAVE_DIR']       = str(OUTPUT_DIR)
env['MEDSAM_VIT_B_CKPT']         = str(MEDSAM_CKPT)

# Also patch the checkpoint_path in the config via env var so we don't edit YAML
# The config reads: checkpoint_path: checkpoints/medsam/medsam_vit_b.pth
# We symlink the Kaggle input file to that location so the relative path resolves.
ckpt_link = REPO_DIR / 'checkpoints' / 'medsam' / 'medsam_vit_b.pth'
ckpt_link.parent.mkdir(parents=True, exist_ok=True)
if not ckpt_link.exists():
    ckpt_link.symlink_to(MEDSAM_CKPT)
    print(f'Symlinked MedSAM checkpoint: {ckpt_link} -> {MEDSAM_CKPT}')
else:
    print(f'MedSAM checkpoint already at: {ckpt_link}')

cmd = [
    sys.executable, '-m', 'src.training.train_component4_lung',
    '--config', str(REPO_DIR / 'configs' / 'component4_lung.yaml'),
]

print('\nStarting Component 4 training...')
print(f'Command: {" ".join(cmd)}')
print('─' * 60)

result = subprocess.run(cmd, env=env, cwd=str(REPO_DIR))

print('─' * 60)
print(f'Training exit code: {result.returncode}')
if result.returncode != 0:
    raise RuntimeError('Component 4 training failed — check output above for the error')

In [ ]:
# ── Cell 5: Verify outputs and print final summary ───────────────────────────
import json
from pathlib import Path

OUTPUT_DIR = Path('/kaggle/working/component4_out')

best_ckpt = OUTPUT_DIR / 'component4_mask_decoder.pt'
last_ckpt = OUTPUT_DIR / 'last_component4_mask_decoder.pt'
history   = OUTPUT_DIR / 'training_history.jsonl'

print('Output files:')
for p in [best_ckpt, last_ckpt, history]:
    if p.exists():
        print(f'  {p.name:<45} {p.stat().st_size // 1024:>8} KB')
    else:
        print(f'  {p.name:<45} MISSING')

# Read training history and print a quick summary
if history.exists():
    records = [json.loads(line) for line in history.read_text().strip().splitlines()]
    print(f'\nTraining history: {len(records)} epochs')
    print(f'{"Epoch":>6} {"Train Dice":>12} {"Val Dice":>10} {"Val IoU":>9}')
    print('─' * 42)
    # Print first 3, every 10th, and last 3
    show = set(range(min(3, len(records))))
    show |= set(range(0, len(records), 10))
    show |= set(range(max(0, len(records) - 3), len(records)))
    for i, rec in enumerate(records):
        if i in show:
            print(f'{rec["epoch"]:>6} {rec["train"]["dice"]:>12.4f} {rec["val"]["dice"]:>10.4f} {rec["val"]["iou"]:>9.4f}')

    best = max(records, key=lambda r: r['val']['dice'])
    print(f'\nBest val Dice: {best["val"]["dice"]:.4f} at epoch {best["epoch"]}')

# Print checkpoint metadata
if best_ckpt.exists():
    import torch
    payload = torch.load(best_ckpt, map_location='cpu')
    print(f'\nCheckpoint metadata:')
    print(f'  epoch      : {payload.get("epoch", "?")}')
    print(f'  best_dice  : {payload.get("best_dice", "?"):.4f}')
    print(f'  model_type : {payload.get("model_type", "?")}')
    print(f'  threshold  : {payload.get("mask_threshold", "?")}')
    print(f'\nNext step: update mask_threshold in configs/component4_lung.yaml')
    print(f'with the "Recommended mask_threshold" printed at the end of Cell 4.')

In [ ]:
# ── Cell 6: Per-dataset val Dice breakdown ───────────────────────────────────
#
# Runs inference on every val sample and reports Dice separately for
# Montgomery and Shenzhen so you can see which dataset benefited more.

import torch
import torch.nn.functional as F
from pathlib import Path
from collections import defaultdict
import sys, os

REPO_DIR   = Path('/kaggle/working/repo')
OUTPUT_DIR = Path('/kaggle/working/component4_out')
sys.path.insert(0, str(REPO_DIR))
os.chdir(str(REPO_DIR))

from src.components.component4_lung import Component4MedSAM
from src.data.component4_lung_dataset import (
    Component4LungDataset, collate_component4_batch, parse_manifest
)
from torch.utils.data import DataLoader

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

best_ckpt = OUTPUT_DIR / 'component4_mask_decoder.pt'
payload = torch.load(best_ckpt, map_location=device)

model = Component4MedSAM(
    backend='auto',
    checkpoint_path=str(REPO_DIR / 'checkpoints' / 'medsam' / 'medsam_vit_b.pth'),
    model_type='vit_b',
    mask_threshold=float(payload.get('mask_threshold', 0.5)),
).to(device)
model.load_decoder_state_dict(payload['decoder_state_dict'])
model.eval()
print(f'Loaded best checkpoint (epoch {payload.get("epoch","?")}, best_dice={payload.get("best_dice","?"):.4f})')

MANIFEST_PATH = OUTPUT_DIR / 'component4_manifest.csv'
val_records = parse_manifest(str(MANIFEST_PATH), repo_root=REPO_DIR, split='val')
val_dataset = Component4LungDataset(val_records, apply_clahe=None, augment=False)
val_loader  = DataLoader(val_dataset, batch_size=2, num_workers=2, shuffle=False,
                         collate_fn=collate_component4_batch)

# Threshold from the sweep (read it from the payload or use best_threshold from above)
threshold = float(payload.get('mask_threshold', 0.5))
print(f'Using threshold: {threshold}')

per_dataset_dice = defaultdict(list)
smooth = 1e-5

with torch.no_grad():
    for batch in val_loader:
        x_3ch   = batch['x_3ch'].to(device)
        targets  = batch['mask'].to(device)
        datasets = batch['dataset']
        logits   = model.forward_logits(x_3ch)
        preds    = (torch.sigmoid(logits) > threshold).float()
        for i, ds in enumerate(datasets):
            p, t = preds[i], targets[i]
            dice = (2 * (p * t).sum() + smooth) / (p.sum() + t.sum() + smooth)
            per_dataset_dice[ds].append(float(dice))

print('\nPer-dataset val Dice:')
print(f'{"Dataset":<15} {"N":>5} {"Mean Dice":>12} {"Min":>8} {"Max":>8}')
print('─' * 52)
all_scores = []
for ds, scores in sorted(per_dataset_dice.items()):
    all_scores.extend(scores)
    print(f'{ds:<15} {len(scores):>5} {sum(scores)/len(scores):>12.4f} {min(scores):>8.4f} {max(scores):>8.4f}')
if all_scores:
    print(f'{"Overall":<15} {len(all_scores):>5} {sum(all_scores)/len(all_scores):>12.4f}')

## Next steps after this notebook

1. **Download** `/kaggle/working/component4_out/component4_mask_decoder.pt`
2. **Upload** it to your `mabdullahi454/tb-pipeline-checkpoints` Kaggle dataset (replace the old file)
3. **Update** `configs/component4_lung.yaml` → `model.mask_threshold` with the value printed at the end of Cell 4
4. **Re-run** `eval_baseline.ipynb` — attach the updated checkpoint dataset — and compare the new Lung Dice and Timika AUROC numbers against the baseline above

If val Dice has **not** improved, check:
- Cell 3 output — did both Montgomery and Shenzhen load? If only one dataset is present, Dice will plateau around 0.65
- Cell 4 epoch log — did val Dice improve monotonically, or did it plateau early? If plateau at epoch 10, the MedSAM checkpoint may not have loaded correctly (training from random weights gives Dice ≈ 0.5 early and never exceeds 0.7)